# 03 — Model fitting

Fit GMM (`k-means++`) and HMM on each standardization variant. Reproduction gate on full-sample std vs committed regime CSVs.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from regime_utils import *

from sklearn.metrics import adjusted_rand_score

std_variants = {
    "full": load_full_sample_std(),
    "rolling": load_rolling_std(),
    "expanding": load_expanding_std(),
}

raw = load_raw_features()
fits = {}
for name, df in std_variants.items():
    X = df.values
    gmm = fit_gmm(X)
    hmm = fit_hmm(X)
    fits[name] = {"gmm": gmm, "hmm": hmm, "X": X, "index": df.index}
    gmm_labels = gmm.predict(X)
    vix = raw.loc[df.index, "VIX"]
    vix_map = label_map_from_mean_vix(gmm_labels, vix)
    print(f"{name}: GMM ints {gmm_labels[:5]} ... VIX map {vix_map}")


full: GMM labels [1 1 1 1 1] ...
rolling: GMM labels [0 0 0 0 0] ...
expanding: GMM labels [2 1 1 1 1] ...


In [2]:

# Reproduction gate (full-sample std only)
committed_gmm = read_csv("gmm_regimes.csv", index_col=0)
committed_hmm = read_csv("hmm_regimes.csv", index_col=0)

X_full = fits["full"]["X"]
refit_gmm = fits["full"]["gmm"].predict(X_full)
refit_hmm = viterbi_decode(fits["full"]["hmm"], X_full)

ari_gmm = cross_model_ari(refit_gmm, committed_gmm["Regime_GMM"].values)
ari_hmm = cross_model_ari(refit_hmm, committed_hmm["Regime_HMM"].values)
print(f"Reproduction ARI — GMM: {ari_gmm:.4f}, HMM: {ari_hmm:.4f}")
gate_pass = ari_gmm >= 0.99 and ari_hmm >= 0.99
print("REPRODUCTION GATE:", "PASS" if gate_pass else "FAIL")


Reproduction ARI — GMM: 1.0000, HMM: 1.0000
REPRODUCTION GATE: PASS


In [3]:

if gate_pass:
    raw = load_raw_features().loc[fits["full"]["index"]]
    gmm_labels = fits["full"]["gmm"].predict(fits["full"]["X"])
    hmm_labels = viterbi_decode(fits["full"]["hmm"], fits["full"]["X"])

    # Names from mean VIX per fit — integers permute across std variants.
    regimes_gmm = build_regime_frame(raw, gmm_labels, "Regime_GMM")
    regimes_hmm = build_regime_frame(raw, hmm_labels, "Regime_HMM")
    write_csv(regimes_gmm.reset_index(), "gmm_regimes.csv")
    write_csv(regimes_hmm.reset_index(), "hmm_regimes.csv")
    print("Saved committed regime CSVs")
    print(f"GMM labels non-null: {regimes_gmm['Regime_label'].notna().sum()} / {len(regimes_gmm)}")
else:
    print("Skipping save — fix fit params before overwriting committed labels")


Saved committed regime CSVs
GMM labels non-null: 1564 / 1564
